# C1, part 2 — source poisoning and propagation

The memory sweep covered one attack surface. These are the other two parts of C1, and
each tests a defence the first one could not.

**Source poisoning (P3.6)** writes fabricated *pages* into the evidence pool. Unlike a
fabricated trajectory, a fabricated page is written about the target claim in detail with
the stance the adversary wants — so it is genuinely relevant, the evaluator scores it
high, and it is *right* to. C2 cannot help, and every cell reports the evaluator's mean
grade on fabricated strips beside real ones so that claim is measured rather than
asserted. Trust is the only defence left, which is why this runner's two configurations
are trust against no trust.

**Propagation (P3.8)** asks whether the damage outlives the fakes. A poisoned record
makes the agent misjudge claim A; A's verdict is stored as experience; that record then
reaches a neighbouring claim B. By then the harmful record is *not* fabricated — it is
the pipeline's own output citing real publishers, so C2 sees a coherent trajectory and C3
sees ordinary domains. Three conditions per pair (absent / clean / corrupted) differing
only in A's verdict make a shift in B attributable.

Both runs reuse the first sweep's cache, so shared work replays from disk.

In [ ]:
import os
import pathlib
import subprocess
import sys

os.chdir("/kaggle/working")
REPO = pathlib.Path("/kaggle/working/VeriMem")
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/TERABYTE05/VeriMem.git", str(REPO)],
        check=True,
    )
os.chdir(REPO)
sys.path.insert(0, str(REPO))

RESULTS = "/kaggle/working/results"
os.environ["VERIMEM_DATA_DIR"] = "/kaggle/working/data"
os.environ["VERIMEM_CACHE_DIR"] = "/kaggle/working/cache"
os.environ["VERIMEM_RESULTS_DIR"] = RESULTS
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## Inputs and the warm cache

In [ ]:
import pathlib
import shutil

DATA = pathlib.Path("/kaggle/working/data/averitec")
TRAJ = pathlib.Path("/kaggle/working/data/trajectories")
for d in (DATA, TRAJ):
    d.mkdir(parents=True, exist_ok=True)

print("attached:", [p.name for p in pathlib.Path("/kaggle/input").iterdir()])

for name, dest in (
    ("dev.json", DATA), ("pool_dev.jsonl", DATA), ("misinfo_list.txt", DATA),
    ("trajectories.jsonl", TRAJ),
):
    if (dest / name).exists():
        continue
    for found in pathlib.Path("/kaggle/input").rglob(name):
        shutil.copy(found, dest / name)
        break
    else:
        raise SystemExit(f"{name} not found among the attached inputs")

for p in sorted(DATA.iterdir()) + sorted(TRAJ.iterdir()):
    print(f"  {p.name:<24} {p.stat().st_size/1e6:7.1f} MB")

restored = 0
for ns in ("llm", "evaluator"):
    dst = pathlib.Path(f"/kaggle/working/cache/{ns}")
    if dst.exists():
        continue
    for src in pathlib.Path("/kaggle/input").rglob(f"cache/{ns}"):
        if src.is_dir():
            shutil.copytree(src, dst)
            n = len(list(dst.rglob("*.json")))
            restored += n
            print(f"restored {n} cached {ns} calls")
            break
if not restored:
    print("WARNING: cold cache. Every call will be recomputed; expect the full runtime.")

In [ ]:
!pip install -q transformers datasets accelerate openai pyyaml python-dotenv 2>&1 | tail -2

## The evaluator

In [ ]:
def find_evaluator():
    for root in (pathlib.Path("/kaggle/working"), pathlib.Path("/kaggle/input")):
        if not root.exists():
            continue
        for p in root.rglob("model/config.json"):
            if (p.parent / "model.safetensors").exists() or (p.parent / "pytorch_model.bin").exists():
                return p.parent
    return None


EVALUATOR = find_evaluator()
if EVALUATOR is None:
    raise SystemExit("no evaluator attached - attach the G1 kernel output")
print("EVALUATOR:", EVALUATOR)

from evaluator.score import T5Scorer

scorer = T5Scorer(EVALUATOR)
s = scorer.score(
    "The Kochi metro opened to the public in 2017.",
    ["The Kochi Metro began commercial operations in June 2017 after several delays.",
     "Penguins are flightless birds found mainly in the southern hemisphere."],
)
print(f"relevant {s[0]:+.3f}   irrelevant {s[1]:+.3f}")
assert s[0] > s[1], "evaluator is not separating - check the model"

## Endpoint

In [ ]:
import shutil
import subprocess
import time
import urllib.request

if shutil.which("zstd") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y zstd",
                   shell=True, capture_output=True)
if shutil.which("ollama") is None:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, capture_output=True, text=True)
OLLAMA_BIN = shutil.which("ollama")
assert OLLAMA_BIN, "ollama did not install"

try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
except Exception:
    subprocess.Popen([OLLAMA_BIN, "serve"], stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(120):
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("ollama did not start")
print("ollama up", flush=True)
subprocess.run([OLLAMA_BIN, "pull", "qwen2.5:7b"], check=True)

os.environ["VERIMEM_API_BASE_URL"] = "http://localhost:11434/v1"
os.environ["VERIMEM_API_MODEL"] = "qwen2.5:7b"
os.environ.setdefault("VERIMEM_API_KEY", "ollama")

COMMON = [
    "--data", "/kaggle/working/data/averitec/dev.json",
    "--store", "/kaggle/working/data/averitec/pool_dev.jsonl",
    "--trajectories", "/kaggle/working/data/trajectories/trajectories.jsonl",
    "--evaluator", str(EVALUATOR),
    "--misinfo", "/kaggle/working/data/averitec/misinfo_list.txt",
    "--targets", "100",
]

## P3.6 — source poisoning, naive adversary

0, 1, 3 and 5 fabricated pages per target. The pool is assembled **in memory** and
discarded; fabricated pages never touch the real pool on disk (rule 10).

In [ ]:
r = subprocess.run(
    [sys.executable, "-u", "-m", "eval.run_source_poisoning", *COMMON,
     "--adversary", "naive", "--pages", "0", "1", "3", "5",
     "--name", "source-poison-naive"],
    env=os.environ,
)
if r.returncode != 0:
    raise SystemExit("source poisoning (naive) failed")

## P3.6 — source poisoning, adaptive adversary

The fabricated pages now carry the domains the clean history trusts most, so C3 has
nothing to object to either. With C2 unable to separate relevance from truth and C3
impersonated, nothing in the pipeline is left — this is the worst case for VeriMem and
it belongs in the report as such.

Two page counts rather than four: the dose curve comes from the naive run, and what is
being asked here is whether the defence survives a stronger attacker at full strength.

In [ ]:
r = subprocess.run(
    [sys.executable, "-u", "-m", "eval.run_source_poisoning", *COMMON,
     "--adversary", "adaptive", "--pages", "0", "5",
     "--name", "source-poison-adaptive"],
    env=os.environ,
)
if r.returncode != 0:
    raise SystemExit("source poisoning (adaptive) failed")

## P3.8 — propagation

One neighbour per target, paired by BM25 similarity relative to the target's own
self-score. Run twice: once with the neighbours undefended, once with C2 and C3 on. The
prediction is that the defences make little difference, because the record carrying the
poison forward is genuine — and if they *do* help, that is a more interesting result than
the one expected.

In [ ]:
for flag, name in ((None, "propagation-undefended"), ("--defences", "propagation-defended")):
    argv = [sys.executable, "-u", "-m", "eval.run_propagation", *COMMON,
            "--rate", "0.2", "--per-target", "1", "--name", name]
    if flag:
        argv.append(flag)
    print(f"\n########## {name} ##########", flush=True)
    r = subprocess.run(argv, env=os.environ)
    if r.returncode != 0:
        raise SystemExit(f"{name} failed")

## Every table, printed to the log

In [ ]:
import json
import pathlib

for f in sorted(pathlib.Path(RESULTS).rglob("metrics.json")):
    d = json.loads(f.read_text())
    name = f.parent.name

    if "source-poison" in name:
        print(f"\n=== {name}: adversary {d.get('adversary')} ===")
        if d.get("attack_domains"):
            print(f"    domains: {', '.join(d['attack_domains'][:6])}")
        print(f"  {'config':<14}{'pages':>6}{'ASR':>8}{'macro-F1':>10}"
              f"{'fake kept':>11}{'grade fake':>12}{'grade real':>12}")
        for c in d["cells"]:
            g, rt = c["grades"], c["retrieval"]
            fk = rt.get("poison_pass_rate")
            print(f"  {c['config']:<14}{c['pages_per_target']:>6}"
                  f"{c['attack_success_rate']:>8.3f}{c['macro_f1']:>10.3f}"
                  f"{'-' if fk is None else format(fk, '.3f'):>11}"
                  f"{'-' if g.get('mean_grade_fake') is None else format(g['mean_grade_fake'], '+.3f'):>12}"
                  f"{'-' if g.get('mean_grade_real') is None else format(g['mean_grade_real'], '+.3f'):>12}")

    elif "propagation" in name:
        print(f"\n=== {name}: defences {d.get('defences_on')} ===")
        print(f"  pairs {d['n_pairs']}   targets flipped {d['n_targets_flipped']}")
        print(f"  propagated {d['n_propagated']}  (rate {d['propagation_rate']})")
        print(f"  collateral {d['n_collateral']}  (rate {d['collateral_rate']})")
        for cond, m in (d.get("neighbours") or {}).items():
            print(f"    {cond:<11} macro-F1 {m['macro_f1']:.3f}  accuracy {m['accuracy']:.3f}")

In [ ]:
import shutil
shutil.rmtree("/kaggle/working/VeriMem/.git", ignore_errors=True)
!du -sh /kaggle/working/cache /kaggle/working/results